# A Catholic Introduction to Artificial Intelligence
## Final Project — Module 6: The Presentation Report
### Classes 11 & 12

---

> *"We become fully human when we become more than human, when we let God bring us beyond ourselves in order to attain the fullest truth of our being. It is precisely this humanity — wounded yet beloved — that must never be replaced or surpassed. We can embrace the technological progress that alleviates suffering and unlocks new possibilities, provided that we do not abandon the very essence of our humanity, namely the capacity for relationship and love."*
> — Pope Leo XIV, *Magnifica Humanitas*, no. 126 & 128

---

## Module Overview

This is the final module of the project. Over five previous modules you have:

- **Module 1** — Explored the dataset; understood what 15 variables mean for 500 homework submissions
- **Module 2** — Investigated patterns; engineered new features; made deliberate decisions about which variables to use
- **Module 3** — Built and evaluated a predictive model; discovered that 81% accuracy can hide near-total failure for the minority class
- **Module 4** — Audited the model for bias; found a 46-percentage-point accuracy gap between the best- and worst-served student tiers
- **Module 5** — Drafted a governance policy; evaluated deployment scenarios; made a recommendation

This module assembles that work into a **presentation-ready final report** — a document that could be presented to a school board, a parent meeting, or a class. It combines polished visualizations, an executive summary, the governance policy, and a final ethical reflection.

**This notebook is your Class 13 presentation.**

---

## Setup: Full Pipeline (Final Version)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, confusion_matrix, roc_auc_score,
    roc_curve, ConfusionMatrixDisplay
)

# ── Presentation styling ──────────────────────────────────────────────
REPORT_STYLE = {
    'font.family': 'sans-serif',
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 9,
    'figure.dpi': 120,
}
plt.rcParams.update(REPORT_STYLE)
sns.set_theme(style='whitegrid')

COLORS = {
    'on_time':  '#4CAF50',
    'late':     '#E57373',
    'flagged':  '#FFB74D',
    'blue':     '#5C8BC7',
    'purple':   '#7986CB',
    'teal':     '#4DB6AC',
    'gray':     '#9E9E9E',
}

TIER_COLORS  = ['#E53935', '#FF8F00', '#43A047', '#1E88E5']
TIER_ORDER   = ['<60%', '60–75%', '75–90%', '>90%']

# ── Data and model ────────────────────────────────────────────────────
df = pd.read_csv('synthetic_homework_dataset.csv',
                 parse_dates=['date_assigned', 'date_submitted'])

df['time_pressure']     = df['difficulty'] / df['days_until_due']
le = LabelEncoder()
df['assignment_type_enc'] = le.fit_transform(df['assignment_type'])
df['prior_tier'] = pd.cut(
    df['prior_completion_rate'],
    bins=[0, 0.60, 0.75, 0.90, 1.01],
    labels=TIER_ORDER
)

FEATURES = [
    'num_questions', 'difficulty', 'days_until_due', 'time_pressure',
    'assignment_type_enc', 'prior_completion_rate',
    'prior_avg_grade', 'prior_avg_homework_time'
]
TARGET = 'completed_on_time'

X, y = df[FEATURES], df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler = StandardScaler()
model  = LogisticRegression(random_state=42, max_iter=1000)
model.fit(scaler.fit_transform(X_train), y_train)

y_pred = model.predict(scaler.transform(X_test))
y_prob = model.predict_proba(scaler.transform(X_test))[:, 1]

results = df.loc[X_test.index].copy()
results['predicted']    = y_pred
results['prob_on_time'] = y_prob
results['correct']      = (results['predicted'] == results[TARGET]).astype(int)

overall_acc = accuracy_score(y_test, y_pred)
auc         = roc_auc_score(y_test, y_prob)
cm          = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
cv_scores   = cross_val_score(
    LogisticRegression(random_state=42, max_iter=1000),
    scaler.fit_transform(X), y, cv=5
)

print('Pipeline ready.')
print(f'Overall accuracy: {overall_acc*100:.1f}%  |  ROC AUC: {auc:.3f}  |  CV: {cv_scores.mean():.3f}')

---

# SECTION 1: EXECUTIVE SUMMARY

*This section is written for a non-technical audience — a school board, a parent meeting, or a colleague who did not take this course. It should be readable without any programming knowledge.*

In [ ]:
# Print the executive summary
THRESH = 0.70
flagged_n      = (y_prob < THRESH).sum()
late_caught    = ((y_prob < THRESH) & (y_test == 0)).sum()
false_flags    = ((y_prob < THRESH) & (y_test == 1)).sum()
total_late     = (y_test == 0).sum()
precision_at_t = late_caught / flagged_n if flagged_n > 0 else 0

tier_accs = results.groupby('prior_tier', observed=True).apply(
    lambda g: g['correct'].mean()
)
max_gap = (tier_accs.max() - tier_accs.min()) * 100

summary = f"""
╔══════════════════════════════════════════════════════════════╗
║            EXECUTIVE SUMMARY                                 ║
║  AI-Assisted Homework Completion Prediction System           ║
╚══════════════════════════════════════════════════════════════╝

WHAT WAS BUILT
A machine learning system was designed and evaluated to predict
whether a student will submit a homework assignment on time.
The system analyzed {len(df)} homework records from {df['student_id'].nunique()}
students across {df['assignment_id'].nunique()} assignments.

WHAT THE SYSTEM CAN DO
  • Overall accuracy:        {overall_acc*100:.1f}%
  • Model quality (ROC AUC): {auc:.3f} (1.0 = perfect, 0.5 = random)
  • At recommended threshold (0.70):
      - Flags {flagged_n} out of 100 students as potentially at risk
      - Of flagged students, {precision_at_t*100:.0f}% are genuinely at risk
      - Catches {late_caught} out of {total_late} actually-late submissions ({late_caught/total_late*100:.0f}%)

WHAT THE SYSTEM CANNOT DO
  • It misses {total_late - late_caught} out of {total_late} actually-late submissions
  • It cannot explain WHY a student is at risk
  • It cannot account for circumstances not in the data
  • It performs significantly worse for students with weaker histories

THE BIAS FINDING
  The system has a {max_gap:.0f}-percentage-point accuracy gap between
  the students it serves best (those with strong prior records)
  and those it serves worst (those with weaker prior records).
  Students who most need support are the ones the model is
  least accurate for — a serious ethical concern.

THE RECOMMENDATION
  [Insert your deployment recommendation from Module 5 here]

GOVERNING PRINCIPLE
  This system may only be used as one input to teacher judgment.
  No automated action, record notation, or disciplinary consequence
  may result from a system prediction without human review.
  The system advises; the teacher decides.
"""
print(summary)

---

# SECTION 2: THE PROJECT JOURNEY

*This section tells the story of what was built and discovered, module by module.*

In [ ]:
# ── FIGURE 1: The Dataset at a Glance ────────────────────────────────
fig = plt.figure(figsize=(15, 9))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)

# 1a. On-time vs late overall
ax1 = fig.add_subplot(gs[0, 0])
counts = df[TARGET].value_counts()
ax1.bar(['On Time', 'Late'], [counts.get(1,0), counts.get(0,0)],
        color=[COLORS['on_time'], COLORS['late']], width=0.5, edgecolor='white')
ax1.set_title('Overall Submission Outcomes')
ax1.set_ylabel('Number of Submissions')
for i, (label, v) in enumerate([('On Time', counts.get(1,0)), ('Late', counts.get(0,0))]):
    ax1.text(i, v + 4, f'{v}\n({v/len(df)*100:.1f}%)', ha='center', fontsize=9)

# 1b. Completion rate by assignment type
ax2 = fig.add_subplot(gs[0, 1])
ct  = df.groupby('assignment_type')[TARGET].mean().sort_values()
ax2.barh(ct.index, ct.values * 100, color=COLORS['blue'], alpha=0.85)
ax2.set_xlabel('On-Time Rate (%)')
ax2.set_title('On-Time Rate by Assignment Type')
ax2.set_xlim(0, 115)
for i, v in enumerate(ct.values):
    ax2.text(v * 100 + 0.5, i, f'{v*100:.1f}%', va='center', fontsize=9)

# 1c. Prior completion rate distribution
ax3 = fig.add_subplot(gs[0, 2])
ax3.hist(df['prior_completion_rate'], bins=15,
         color=COLORS['purple'], edgecolor='white', alpha=0.85)
ax3.set_xlabel('Prior Completion Rate')
ax3.set_ylabel('Number of Records')
ax3.set_title('Distribution of Students\nby Prior Completion Rate')
ax3.axvline(df['prior_completion_rate'].mean(), color='red', linestyle='--',
            linewidth=1.5, label=f"Mean: {df['prior_completion_rate'].mean():.2f}")
ax3.legend()

# 1d. Late rate by prior tier
ax4 = fig.add_subplot(gs[1, 0])
tier_late = df.groupby('prior_tier', observed=True)[TARGET].apply(lambda x: 1-x.mean())
ax4.bar(TIER_ORDER, [tier_late.get(t, 0)*100 for t in TIER_ORDER],
        color=TIER_COLORS, edgecolor='white', alpha=0.85)
ax4.set_xlabel('Prior Completion Tier')
ax4.set_ylabel('Actual Late Rate (%)')
ax4.set_title('Who Is Actually Late?\nBy Prior Completion Tier')
for i, t in enumerate(TIER_ORDER):
    v = tier_late.get(t, 0)
    n = (df['prior_tier']==t).sum()
    ax4.text(i, v*100 + 0.5, f'{v*100:.0f}%\n(n={n})', ha='center', fontsize=8)

# 1e. Difficulty vs completion rate
ax5 = fig.add_subplot(gs[1, 1])
diff_comp = df.groupby('difficulty')[TARGET].mean()
diff_n    = df.groupby('difficulty').size()
ax5.bar(diff_comp.index, diff_comp.values * 100,
        color=COLORS['teal'], edgecolor='white', alpha=0.85)
ax5.set_xlabel('Difficulty Level (1=Easy, 5=Hard)')
ax5.set_ylabel('On-Time Completion Rate (%)')
ax5.set_title('On-Time Rate by Difficulty')
ax5.set_ylim(0, 115)
for i, (diff, v) in enumerate(diff_comp.items()):
    ax5.text(diff, v*100+1, f'{v*100:.0f}%', ha='center', fontsize=9)

# 1f. Correlation with target
ax6 = fig.add_subplot(gs[1, 2])
numeric_cols = ['num_questions','difficulty','days_until_due','time_pressure',
                'prior_completion_rate','prior_avg_grade','prior_avg_homework_time']
corrs = df[numeric_cols + [TARGET]].corr()[TARGET].drop(TARGET).sort_values()
bar_colors = [COLORS['on_time'] if v > 0 else COLORS['late'] for v in corrs.values]
ax6.barh(corrs.index, corrs.values, color=bar_colors, alpha=0.85)
ax6.axvline(0, color='black', linewidth=0.8)
ax6.set_xlabel('Correlation with On-Time Submission')
ax6.set_title('What Correlates with\nOn-Time Submission?')

fig.suptitle('Figure 1: The Dataset at a Glance', fontsize=14, fontweight='bold', y=1.01)
plt.savefig('fig1_dataset.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure 1 saved.')

In [ ]:
# ── FIGURE 2: What the Model Learned ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 2a. Logistic regression coefficients
coef_df = pd.DataFrame({
    'Feature': FEATURES,
    'Coefficient': model.coef_[0]
}).sort_values('Coefficient')

bar_colors = [COLORS['on_time'] if v > 0 else COLORS['late']
              for v in coef_df['Coefficient']]
axes[0].barh(coef_df['Feature'], coef_df['Coefficient'],
             color=bar_colors, alpha=0.85)
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_xlabel('Coefficient Value')
axes[0].set_title('What the Model Learned\n'
                  'Green = pushes toward "on time", Red = pushes toward "late"')
for bar, v in zip(axes[0].patches, coef_df['Coefficient']):
    axes[0].text(v + (0.01 if v >= 0 else -0.01),
                 bar.get_y() + bar.get_height()/2,
                 f'{v:+.3f}', va='center',
                 ha='left' if v >= 0 else 'right', fontsize=8)

# 2b. Decision tree feature importances
dt = DecisionTreeClassifier(max_depth=4, random_state=42)
dt.fit(X_train, y_train)
imp_df = pd.DataFrame({
    'Feature': FEATURES,
    'Importance': dt.feature_importances_
}).sort_values('Importance')

axes[1].barh(imp_df['Feature'], imp_df['Importance'],
             color=COLORS['purple'], alpha=0.85)
axes[1].set_xlabel('Feature Importance')
axes[1].set_title('Decision Tree: Feature Importance\n'
                  '(How much does each feature contribute?)')

fig.suptitle('Figure 2: What the Model Learned', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig2_model.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure 2 saved.')

In [ ]:
# ── FIGURE 3: Why Accuracy Misleads ──────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 3a. Baseline confusion matrix
baseline = np.ones(len(y_test), dtype=int)
ConfusionMatrixDisplay(
    confusion_matrix(y_test, baseline),
    display_labels=['Late', 'On Time']
).plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title(f'Baseline: Always Predict On Time\n'
                  f'Accuracy: {accuracy_score(y_test, baseline)*100:.1f}%  |  '
                  f'Late caught: 0/{(y_test==0).sum()}')

# 3b. Our model confusion matrix
ConfusionMatrixDisplay(
    cm, display_labels=['Late', 'On Time']
).plot(ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title(f'Our Model (threshold = 0.50)\n'
                  f'Accuracy: {overall_acc*100:.1f}%  |  '
                  f'Late caught: {tn}/{tn+fp}')

# 3c. ROC curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[2].plot(fpr, tpr, color=COLORS['blue'], linewidth=2,
             label=f'Model (AUC = {auc:.3f})')
axes[2].plot([0,1], [0,1], 'k--', linewidth=1, label='Random (AUC = 0.500)')
axes[2].set_xlabel('False Positive Rate')
axes[2].set_ylabel('True Positive Rate')
axes[2].set_title('ROC Curve\n(Top-left corner = better)')
axes[2].legend()

fig.suptitle('Figure 3: Why Accuracy Alone Is Not Enough', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig3_evaluation.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure 3 saved.')

In [ ]:
# ── FIGURE 4: The Bias Audit ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 4a. Accuracy by tier
tier_acc_vals = results.groupby('prior_tier', observed=True)['correct'].mean()
acc_vals = [tier_acc_vals.get(t, 0) * 100 for t in TIER_ORDER]

bars = axes[0].bar(TIER_ORDER, acc_vals, color=TIER_COLORS, edgecolor='white', alpha=0.9)
axes[0].axhline(overall_acc*100, color='black', linestyle='--', linewidth=1.5,
                label=f'Overall: {overall_acc*100:.1f}%')
axes[0].set_xlabel('Prior Completion Rate Tier')
axes[0].set_ylabel('Model Accuracy (%)')
axes[0].set_title('Model Accuracy by\nPrior Completion Tier')
axes[0].set_ylim(0, 115)
axes[0].legend()
for bar, v in zip(bars, acc_vals):
    axes[0].text(bar.get_x()+bar.get_width()/2, v+1.5,
                 f'{v:.0f}%', ha='center', fontsize=10, fontweight='bold')

# 4b. Training data representation
train_df = df.loc[X_train.index].copy()
train_df['prior_tier'] = pd.cut(
    train_df['prior_completion_rate'],
    bins=[0, 0.60, 0.75, 0.90, 1.01], labels=TIER_ORDER
)
train_counts = train_df['prior_tier'].value_counts().reindex(TIER_ORDER, fill_value=0)
axes[1].bar(TIER_ORDER, train_counts.values, color=TIER_COLORS, edgecolor='white', alpha=0.9)
axes[1].set_xlabel('Prior Completion Rate Tier')
axes[1].set_ylabel('Training Records')
axes[1].set_title('Training Data Representation\nby Tier (Root Cause of Bias)')
for i, v in enumerate(train_counts.values):
    axes[1].text(i, v+1, str(v), ha='center', fontsize=10, fontweight='bold')

# 4c. Scatter: prior rate vs. model accuracy per student
student_audit = results.groupby('student_id').apply(
    lambda g: pd.Series({
        'accuracy': g['correct'].mean(),
        'prior_rate': g['prior_completion_rate'].mean(),
        'late_rate': (g['completed_on_time']==0).mean(),
        'n': len(g)
    })
).reset_index()
student_audit = student_audit[student_audit['n'] >= 2]

sc = axes[2].scatter(
    student_audit['prior_rate'],
    student_audit['accuracy'],
    c=student_audit['late_rate'],
    cmap='RdYlGn_r', vmin=0, vmax=0.6,
    s=70, alpha=0.8, edgecolors='white', linewidths=0.5
)
plt.colorbar(sc, ax=axes[2], label='Actual late rate')
axes[2].axhline(overall_acc, color='navy', linestyle='--',
                linewidth=1, label=f'Overall: {overall_acc*100:.1f}%')
axes[2].set_xlabel('Prior Completion Rate')
axes[2].set_ylabel('Model Accuracy (per student)')
axes[2].set_title('Students With Lower Prior Rates\nHave Worse Model Performance')
axes[2].legend()

fig.suptitle('Figure 4: The Bias Audit — A 46-Point Accuracy Gap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig4_bias.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure 4 saved.')

In [ ]:
# ── FIGURE 5: Deployment Threshold Analysis ───────────────────────────
thresholds      = np.arange(0.50, 0.96, 0.01)
flagged_pcts    = []
late_recalls    = []
precisions      = []
false_flag_rates = []
total_late      = (y_test == 0).sum()
total_ontime    = (y_test == 1).sum()

for t in thresholds:
    preds = (y_prob >= t).astype(int)
    flag_n   = (y_prob < t).sum()
    lc       = ((y_prob < t) & (y_test == 0)).sum()
    ff       = ((y_prob < t) & (y_test == 1)).sum()
    flagged_pcts.append(flag_n / len(y_test))
    late_recalls.append(lc / total_late)
    precisions.append(lc / flag_n if flag_n > 0 else 0)
    false_flag_rates.append(ff / total_ontime)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: the trade-off curves
axes[0].plot(thresholds, [r*100 for r in late_recalls],
             color=COLORS['on_time'], linewidth=2, label='Late students caught (%)')
axes[0].plot(thresholds, [r*100 for r in false_flag_rates],
             color=COLORS['late'], linewidth=2, label='On-time students wrongly flagged (%)')
axes[0].plot(thresholds, [p*100 for p in precisions],
             color=COLORS['blue'], linewidth=2, linestyle='--', label='Precision of flags (%)')
axes[0].axvline(0.70, color='black', linestyle=':', linewidth=1.5, label='Recommended threshold (0.70)')
axes[0].set_xlabel('Decision Threshold')
axes[0].set_ylabel('%')
axes[0].set_title('The Fundamental Trade-Off:\nCatching More Late Students Means More False Flags')
axes[0].legend(fontsize=8)
axes[0].set_ylim(-5, 105)

# Right: what happens at the recommended threshold
t_rec = 0.70
flagged_t = (y_prob < t_rec).sum()
caught_t  = ((y_prob < t_rec) & (y_test == 0)).sum()
ff_t      = ((y_prob < t_rec) & (y_test == 1)).sum()
missed_t  = total_late - caught_t

categories = ['Correctly\ncaught late', 'Wrongly\nflagged', 'Late students\nmissed', 'Correctly\non time']
values     = [caught_t, ff_t, missed_t, (y_test==1).sum() - ff_t]
colors_bar = [COLORS['on_time'], COLORS['flagged'], COLORS['late'], COLORS['teal']]

axes[1].bar(categories, values, color=colors_bar, edgecolor='white', alpha=0.9)
axes[1].set_ylabel('Number of Students')
axes[1].set_title(f'What Happens at the Recommended Threshold (0.70)\n'
                  f'Out of 100 test students')
for i, v in enumerate(values):
    axes[1].text(i, v + 0.3, str(v), ha='center', fontsize=11, fontweight='bold')

fig.suptitle('Figure 5: Choosing a Deployment Threshold', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig5_threshold.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure 5 saved.')

---

# SECTION 3: KEY FINDINGS SUMMARY

*A structured summary of the five most important findings from the project.*

In [ ]:
tier_accs_dict = results.groupby('prior_tier', observed=True)['correct'].mean().to_dict()

print('=' * 65)
print('KEY FINDINGS')
print('=' * 65)
print()
print('FINDING 1: Past behavior is the strongest predictor')
print(f'  prior_completion_rate has the largest model coefficient (+{model.coef_[0][5]:.3f})')
print(f'  Students with prior rates >90% are late only {df[df["prior_tier"]==">90%"][TARGET].apply(lambda x:1-x).mean()*100:.0f}% of the time')
print(f'  Students with prior rates <75% are late {df[df["prior_tier"].isin(["<60%","60–75%"])][TARGET].apply(lambda x:1-x).mean()*100:.0f}% of the time')
print()
print('FINDING 2: Accuracy overstates performance')
print(f'  A model that always predicts "on time" scores {df[TARGET].mean()*100:.1f}% accuracy')
print(f'  Our model scores {overall_acc*100:.1f}% — only marginally better')
print(f'  At default threshold, model catches only {tn/(tn+fp)*100:.0f}% of late submissions')
print(f'  The ROC AUC of {auc:.3f} shows the model has real but limited discriminative ability')
print()
print('FINDING 3: The model fails the students who need it most')
print('  Accuracy by prior completion tier:')
for t in TIER_ORDER:
    acc = tier_accs_dict.get(t, float('nan'))
    n   = (results['prior_tier'] == t).sum()
    print(f'    {t:<8}: {acc*100:.0f}%  (n={n})')
print(f'  Accuracy gap: {max_gap:.0f} percentage points')
print()
print('FINDING 4: The bias has a structural root cause')
print(f'  Students with prior rates <75% = {(train_df["prior_tier"].isin(["<60%","60–75%"])).sum()} training records ({(train_df["prior_tier"].isin(["<60%","60–75%"])).mean()*100:.0f}% of training data)')
print('  Underrepresentation → model cannot learn their patterns')
print('  Removing history features does not resolve the gap')
print()
print('FINDING 5: Threshold choice has major ethical implications')
print(f'  At 0.50: catches {tn}/{total_late} late ({tn/total_late*100:.0f}%) — almost useless')
print(f'  At 0.70: catches {late_caught}/{total_late} late ({late_caught/total_late*100:.0f}%) with {false_flags} false flags — reasonable tradeoff')
print(f'  At 0.85: catches {((y_prob<0.85)&(y_test==0)).sum()}/{total_late} late — but {((y_prob<0.85)&(y_test==1)).sum()} false flags')
print('  No threshold is both highly sensitive and highly precise')
print()
print('=' * 65)

---

# SECTION 4: THE GOVERNANCE POLICY

*Paste the completed governance policy from Module 5 below, replacing the template text with your final decisions.*

## Data Governance Policy — Homework Completion Prediction System
**Version:** 1.0 — Final  
**Author:** [Your name]  
**Date:** [Today's date]

---

*[Paste your completed governance policy from Module 5 here. Replace all bracketed placeholders with your actual decisions.]*

---

**Policy summary table:**

| Element | Your Decision |
|---|---|
| Deployment scenario | [A / B / C / D] |
| Decision threshold | [0.50 / 0.70 / 0.80 / 0.85 / other] |
| Who sees predictions | [your access structure] |
| Automated actions permitted? | [Yes / No — explain] |
| Student notified? | [Yes / No — explain] |
| Appeal mechanism | [describe] |
| Suspension trigger | [your metric thresholds] |
| Primary Catholic principle | [name the principle and cite the source] |

---

# SECTION 5: ETHICAL SYNTHESIS

*This section connects the technical findings to Catholic social teaching. It should be written in your own voice and reflect genuine engagement with both the data and the tradition.*

## 5.1 What the Data Says About Human Dignity

The most important finding of this project is not statistical. It is human.

The students the model serves worst are the students the model should most want to help — those with lower prior completion rates, higher late rates, and greater need for early support. The model has learned to be confident about students who need little intervention, and uncertain about students who need the most.

This is not a quirk of this particular model. It is a structural pattern in AI systems generally: models learn from data, and data reflects the world as it is, including its inequalities. Students who have struggled in the past have fewer successful examples in the training data. The model has less to learn from. The gap between what the model can offer and what these students need is not an engineering failure — it is a consequence of trying to reduce human complexity to a prediction.

The Compendium of the Social Doctrine of the Church (no. 105) states that human dignity does not depend on what a person achieves or produces. *Magnifica Humanitas* (no. 99) insists that AI systems do not possess moral conscience, do not mature through relationships, and do not know from within what it means to struggle, fail, recover, and change. A system that reduces a student to a set of 15 variables captures nothing of what actually matters about them.

---

## 5.2 The Five Catholic Principles — Applied

Write one paragraph for each principle below, connecting it directly to a specific finding from this project.

### Human Dignity (*Imago Dei*)
*[Connect to a specific finding — e.g., the 46-point accuracy gap, or the impossibility of capturing a student's full circumstances in 15 variables]*

---

### The Common Good
*[Connect to who benefits from this system and who does not — e.g., the model works best for students with strong records who need it least]*

---

### Subsidiarity
*[Connect to the governance decisions — e.g., why human oversight at the teacher level must remain; why no automated actions are permissible]*

---

### Solidarity and the Preferential Option
*[Connect to the bias finding — the students served worst are those most in need; what solidarity demands of the school community]*

---

### Accountability
*[Connect to the moral responsibility gap — when the system makes a wrong prediction that leads to a wrong teacher response, who is responsible?]*

---

## 5.3 The Central Question

*Magnifica Humanitas* (no. 129) asks a question that runs through the entire course and this entire project:

> *"Does AI make human life on earth 'more human' in every aspect of that life? Does it make it more worthy of man?"*

Based on everything this project has found, answer that question honestly for this specific system — not for AI in general, but for this model, deployed in this context, with these results.

*[Write your answer here — 2 to 4 paragraphs. Be specific. Reference at least two findings from the project and at least two teachings from Magnifica Humanitas or the Compendium.]*

---

# SECTION 6: THE FINAL REFLECTION

*This section is personal. It asks you to reflect on what you have learned — not only about AI, but about yourself, about education, and about what it means to build technology responsibly.*

## A Letter to a Future Student

*Write a letter — 2 to 3 paragraphs — addressed to a student who will take this course next year. Tell them:*

1. *The single most important thing you learned about AI that you did not know before*
2. *The moment in this project that surprised you most — technically or ethically*
3. *One piece of advice for navigating a world in which AI systems make consequential decisions about people*

*Write in your own voice. This is not a technical document. It is a reflection on what the technology means for the kind of person you want to be.*

---

*[Write your letter here]*

---

# SECTION 7: PROJECT SUMMARY TABLE

*A single-page summary of the complete project for the Class 13 presentation.*

In [ ]:
# ── FIGURE 6: Project Summary Dashboard ──────────────────────────────
fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.55, wspace=0.40)

# Row 1: The headline numbers
for col, (title, val, sub) in enumerate([
    ('Dataset Size', f'{len(df):,}\nrecords', f'{df.student_id.nunique()} students · {df.assignment_id.nunique()} assignments'),
    ('Overall Accuracy', f'{overall_acc*100:.1f}%', f'ROC AUC = {auc:.3f}'),
    ('Accuracy Gap', f'{max_gap:.0f}pp', 'Between best- and worst-served tiers'),
]):
    ax = fig.add_subplot(gs[0, col])
    ax.text(0.5, 0.55, val, ha='center', va='center',
            transform=ax.transAxes,
            fontsize=22, fontweight='bold',
            color='#1A3A5C')
    ax.text(0.5, 0.18, sub, ha='center', va='center',
            transform=ax.transAxes,
            fontsize=9, color='#555555')
    ax.set_title(title, fontsize=11, fontweight='bold', pad=8)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.axis('off')
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor('#DDDDDD')

# Row 2: Bias chart + threshold trade-off
ax4 = fig.add_subplot(gs[1, :2])
tier_acc_for_plot = [results[results['prior_tier']==t]['correct'].mean()*100 for t in TIER_ORDER]
tier_n  = [(results['prior_tier']==t).sum() for t in TIER_ORDER]
bars = ax4.bar(TIER_ORDER, tier_acc_for_plot, color=TIER_COLORS, edgecolor='white', alpha=0.9)
ax4.axhline(overall_acc*100, color='black', linestyle='--', linewidth=1.5,
            label=f'Overall accuracy: {overall_acc*100:.1f}%')
ax4.set_title('The Core Bias Finding: Model Accuracy by Prior Completion Rate Tier')
ax4.set_ylabel('Accuracy (%)')
ax4.set_ylim(0, 115)
ax4.legend()
for bar, v, n in zip(bars, tier_acc_for_plot, tier_n):
    ax4.text(bar.get_x()+bar.get_width()/2, v+1.5,
             f'{v:.0f}%\n(n={n})', ha='center', fontsize=9, fontweight='bold')

ax5 = fig.add_subplot(gs[1, 2])
recall_vals = [((y_prob<t)&(y_test==0)).sum()/total_late*100 for t in [0.5,0.7,0.8,0.85,0.9]]
ff_vals     = [((y_prob<t)&(y_test==1)).sum()/total_ontime*100 for t in [0.5,0.7,0.8,0.85,0.9]]
ax5.plot([0.5,0.7,0.8,0.85,0.9], recall_vals, 'o-',
         color=COLORS['on_time'], linewidth=2, label='Late caught')
ax5.plot([0.5,0.7,0.8,0.85,0.9], ff_vals, 's-',
         color=COLORS['late'], linewidth=2, label='False flags')
ax5.axvline(0.70, color='black', linestyle=':', linewidth=1.5)
ax5.set_xlabel('Threshold')
ax5.set_ylabel('%')
ax5.set_title('Threshold Trade-Off')
ax5.legend(fontsize=8)

# Row 3: Text summaries
findings_text = (
    "KEY FINDINGS:\n"
    f"1. prior_completion_rate is the strongest\n   predictor (coef = +{model.coef_[0][5]:.3f})\n"
    f"2. At 0.5 threshold: catches only {tn/total_late*100:.0f}%\n   of late submissions\n"
    f"3. At 0.70 threshold: catches {late_caught/total_late*100:.0f}% with\n   {false_flags} false flags out of {total_ontime} on-time\n"
    f"4. Bias gap: {max_gap:.0f}pp between tiers\n"
    f"5. Root cause: underrepresentation in\n   training data"
)
governance_text = (
    "GOVERNANCE DECISIONS:\n"
    "• [Deployment scenario: A/B/C/D]\n"
    "• [Threshold: 0.70 recommended]\n"
    "• No automated actions permitted\n"
    "• Human override always available\n"
    "• [Student notification: yes/no]\n"
    "• Suspension trigger: gap > 30pp\n"
    "• Annual bias audit required"
)
teaching_text = (
    "CATHOLIC PRINCIPLES APPLIED:\n"
    "• Human dignity: model cannot capture\n"
    "  a student's full circumstances\n"
    "• Common good: system serves those\n"
    "  with strong records most; reform needed\n"
    "• Subsidiarity: teacher judgment\n"
    "  must remain; no automated action\n"
    "• Solidarity: worst-served students\n"
    "  are the most vulnerable\n"
    "• Accountability: teacher, not model,\n"
    "  bears responsibility for decisions"
)

for col, (title, text, color) in enumerate([
    ('Technical Summary', findings_text, '#E3F2FD'),
    ('Governance Summary', governance_text, '#E8F5E9'),
    ('Ethical Summary', teaching_text, '#FFF8E1'),
]):
    ax = fig.add_subplot(gs[2, col])
    ax.set_facecolor(color)
    ax.text(0.05, 0.95, text, ha='left', va='top',
            transform=ax.transAxes,
            fontsize=8.5, fontfamily='monospace',
            verticalalignment='top')
    ax.set_title(title, fontsize=10, fontweight='bold', pad=6)
    ax.axis('off')

fig.suptitle(
    'AI-Assisted Homework Completion Prediction System — Final Project Summary',
    fontsize=14, fontweight='bold', y=1.01
)
plt.savefig('fig6_summary_dashboard.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure 6 (summary dashboard) saved.')

In [ ]:
# Print the complete project metrics for reference
print('=' * 65)
print('COMPLETE PROJECT METRICS REFERENCE')
print('=' * 65)
print()
print('THE DATA')
print(f'  Total records:             {len(df)}')
print(f'  Unique students:           {df.student_id.nunique()}')
print(f'  Unique assignments:        {df.assignment_id.nunique()}')
print(f'  Overall on-time rate:      {df[TARGET].mean()*100:.1f}%')
print(f'  Assignment types:          {sorted(df.assignment_type.unique())}')
print()
print('THE MODEL')
print(f'  Algorithm:                 Logistic Regression')
print(f'  Features:                  {len(FEATURES)}')
print(f'  Training records:          {len(X_train)}')
print(f'  Test records:              {len(X_test)}')
print(f'  Test accuracy:             {overall_acc*100:.1f}%')
print(f'  ROC AUC:                   {auc:.3f}')
print(f'  5-fold CV accuracy:        {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')
print(f'  Strongest feature:         prior_completion_rate (coef = +{model.coef_[0][5]:.3f})')
print()
print('ERROR ANALYSIS (default threshold 0.50)')
print(f'  True negatives  (late → late):     {tn}')
print(f'  False positives (late → on-time):  {fp}  ← missed late students')
print(f'  False negatives (on-time → late):  {fn}  ← wrongly flagged')
print(f'  True positives  (on-time → on-time): {tp}')
print(f'  Late recall:               {tn/(tn+fp)*100:.1f}%')
print()
print('AT RECOMMENDED THRESHOLD (0.70)')
print(f'  Students flagged:          {late_caught+false_flags} / {len(X_test)} ({(late_caught+false_flags)/len(X_test)*100:.0f}%)')
print(f'  Late students caught:      {late_caught} / {total_late} ({late_caught/total_late*100:.0f}%)')
print(f'  False flags:               {false_flags} ({false_flags/(len(X_test)-total_late)*100:.0f}% of on-time students)')
print(f'  Precision at 0.70:         {precision_at_t*100:.0f}%')
print()
print('BIAS AUDIT')
for t in TIER_ORDER:
    acc  = tier_accs_dict.get(t, float('nan'))
    n    = (results['prior_tier'] == t).sum()
    late = (df['prior_tier'] == t).apply(lambda x: x)
    lr_t = df[df['prior_tier']==t][TARGET].apply(lambda x: 1-x).mean()
    print(f'  {t:<8}: accuracy={acc*100:.0f}%  actual_late_rate={lr_t*100:.0f}%  n_test={n}')
print(f'  Max accuracy gap:          {max_gap:.0f} percentage points')
print('=' * 65)

---

# Presentation Checklist

Before your Class 13 presentation, confirm that this notebook contains:

**Technical components**
- [ ] All six figures generated and displaying correctly
- [ ] Executive summary with your actual deployment recommendation filled in
- [ ] Key findings section complete
- [ ] Complete metrics reference generated

**Written components**
- [ ] Completed governance policy from Module 5 (all placeholders replaced)
- [ ] Policy summary table filled in
- [ ] Five Catholic principles paragraphs written (one per principle)
- [ ] Answer to the central question (*Magnifica Humanitas* no. 129)
- [ ] Letter to a future student written

**Presentation preparation**
- [ ] You can explain Figure 1 (the dataset) in 60 seconds without notes
- [ ] You can explain the bias finding (Figure 4) and why it matters
- [ ] You can defend your governance policy decision with two reasons
- [ ] You have a personal reflection ready for the closing question

---

> *"Like Nehemiah, we too are called to unite listening and courage, prayer and responsibility, so that, even when a technocratic mentality or partisan interests seem to prevail, the human city may become a more fitting place to live."*
> — Pope Leo XIV, *Magnifica Humanitas*, no. 241
>
> You built something. You evaluated it honestly. You found its limits and named them clearly. You drafted rules to constrain it in the service of the people it affects. That is not a small thing. That is what it means to be a builder of Jerusalem rather than an architect of Babel.